# Guide 2: Tokenization: how text becomes numbers

*How LLMs work* · [Read the guide](https://www.vinaygautam.com/blog/how-llms-work/02-tokenization) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drvgautam/primer-on-aiengineering/blob/main/code/notebooks/02-tokenizer.ipynb)

This notebook holds every code block from Guide 2, in the guide's order: 9 code cells, 9 of which print an output the guide quotes. Each cell is preceded by the guide paragraph that introduces it, so you can keep the guide open beside the notebook. The saved outputs are the guide's; `tools/verify_notebooks.py` checks they still match.

Run the setup cell first, then the cells in order: like the guide's blocks, they share variables.

**Sections of the guide**

- The unit a model reads — no code
- Two bad extremes — 2 code cells
- Learning the vocabulary from data: byte-pair encoding — 1 code cell
- Encoding text the tokenizer has never seen — 1 code cell
- Bytes: the fallback that never fails — 1 code cell
- A real tokenizer on a real sentence — 2 code cells
- Where tokenization shows through — 1 code cell
- From tokens to training examples — 1 code cell

In [1]:
# Setup: run this cell first. It makes the repository's code/ folder the working directory, so
# every block below runs exactly as printed in the guide ("run from the repository root").
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/drvgautam/primer-on-aiengineering"
if "google.colab" in sys.modules and not any((p / "code" / "notebooks").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/primer-on-aiengineering"], check=True)
    os.chdir("/content/primer-on-aiengineering")
    pins = [l.strip() for l in open("code/requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins,
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], check=True)

here = Path.cwd().resolve()
code_dir = next(p / "code" for p in [here, *here.parents] if (p / "code" / "notebooks").is_dir())
os.chdir(code_dir)

# The guide's outputs were produced with these versions, on CPU. Others may differ in the last digits.
import numpy, tiktoken, torch
pins = dict(l.strip().split("==") for l in open("requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch"))
for name, mod in (("numpy", numpy), ("tiktoken", tiktoken), ("torch", torch)):
    have = mod.__version__.split("+")[0]
    note = "" if have == pins[name] else f"  <- the guide used {pins[name]}; outputs may differ slightly (on Colab, restart the runtime and rerun this cell)"
    print(f"{name:9} {have}{note}")
print("working directory:", Path(code_dir.parent.name) / "code")

# %%gpt2 marks blocks the guide shows but doesn't run, because they need GPT-2 small's 548 MB weights.
# To run them, download the weights as series/running-example.py explains, then set:
#     os.environ["PRIMER_RUN_GPT2"] = "1"
from IPython.core.magic import register_cell_magic

@register_cell_magic
def gpt2(line, cell):
    if os.environ.get("PRIMER_RUN_GPT2") == "1":
        get_ipython().run_cell(cell)
    else:
        print("Skipped: needs GPT-2 weights (see the setup cell to enable).")


numpy     2.2.6
tiktoken  0.14.0
torch     2.14.1
working directory: primer-on-aiengineering/code


## Two bad extremes

Take Guide 1's toy text and count it both ways. The code in this guide is plain Python until we reach GPT-2, in blocks that run in order and share variables, like notebook cells.

In [2]:
text = ("the cat sat on the mat and the dog sat on the log and the "
        "cat saw the dog then the dog ran to the cat and the cat ran on the mat")

chars = list(text)
words = text.split()
print(len(chars), "characters,", len(set(chars)), "distinct")
print(len(words), "words,", len(set(words)), "distinct")

128 characters, 15 distinct
33 words, 12 distinct


Words make the sequence about four times shorter, 33 tokens against 128. The vocabulary, though, is now whatever words happened to appear, and that breaks the moment someone types a new one. Here's a word-level tokenizer built from the toy text, given a sentence that ends in a word it has never seen:

In [3]:
vocab = sorted(set(words)) + ["<unk>"]

def encode_words(s):
    return [vocab.index(w) if w in vocab else vocab.index("<unk>") for w in s.split()]

def decode_words(token_ids):
    return " ".join(vocab[i] for i in token_ids)

token_ids = encode_words("the cat sat on the rug")
print(token_ids)
print(decode_words(token_ids))

[9, 1, 7, 5, 9, 12]
the cat sat on the <unk>


## Learning the vocabulary from data: byte-pair encoding

The trainer below is about 30 lines. `merge_pair` does the gluing inside one word. `train_bpe` counts every adjacent pair, picks the most frequent, merges it everywhere, and prints how many tokens the toy text has afterwards.

In [4]:
import re
from collections import Counter

def merge_pair(word, a, b):
    out, i = [], 0
    while i < len(word):
        if word[i:i + 2] == (a, b):   # the pair: glue it into one token
            out.append(a + b)
            i += 2
        else:
            out.append(word[i])
            i += 1
    return tuple(out)

def train_bpe(text, num_merges):
    # pre-tokenize: each word keeps its leading space; Counter maps word -> how often
    words = Counter(tuple(w) for w in re.findall(r" ?[a-z]+", text))
    merges = []
    for step in range(1, num_merges + 1):
        pairs = Counter()
        for word, freq in words.items():
            for pair in zip(word, word[1:]):
                pairs[pair] += freq
        if not pairs:                 # every word is already one token
            break
        (a, b), count = max(pairs.items(), key=lambda kv: (kv[1], kv[0]))
        merges.append((a, b))
        merged = Counter()
        for word, freq in words.items():
            merged[merge_pair(word, a, b)] += freq
        words = merged
        n = sum(len(word) * freq for word, freq in words.items())  # sequence length
        print(f"{step:2d}  {a!r:>5} + {b!r:<5}  seen {count:2d} times  {n:3d} tokens")
    return merges

print(len(text), "tokens at the start")
merges = train_bpe(text, 10)
vocab = sorted(set(text)) + [a + b for a, b in merges]
print(len(vocab), "vocabulary entries")

128 tokens at the start
 1    't' + 'h'    seen 11 times  117 tokens
 2   'th' + 'e'    seen 11 times  106 tokens
 3    ' ' + 'the'  seen 10 times   96 tokens
 4    'a' + 't'    seen  8 times   88 tokens
 5    'a' + 'n'    seen  5 times   83 tokens
 6    'o' + 'g'    seen  4 times   79 tokens
 7    'c' + 'at'   seen  4 times   75 tokens
 8    ' ' + 'cat'  seen  4 times   71 tokens
 9    'o' + 'n'    seen  3 times   68 tokens
10    'd' + 'og'   seen  3 times   65 tokens
25 vocabulary entries


## Encoding text the tokenizer has never seen

Training left us an ordered list of merges. To encode new text, pre-tokenize it the same way, except that the encoder accepts any non-space characters so we can feed it letters the trainer never saw. Then split each word into characters, and replay the merges in the order they were learned. The order matters because later merges are built from earlier ones. Merge 7 glues `c` to `at`, and there is no `at` until merge 4 has run.

In [5]:
def encode(text):
    tokens = []
    for w in re.findall(r" ?\S+", text):   # any run of non-space characters
        word = tuple(w)
        for a, b in merges:        # replay in the order learned
            word = merge_pair(word, a, b)
        tokens.extend(word)
    return tokens

tokens = encode("the cats")
print(tokens, [vocab.index(t) for t in tokens])
tokens = encode("the bat")
print(tokens, "not in vocab:", [t for t in tokens if t not in vocab])
print(len(encode("the cat sat on the mat")), "tokens")

['the', ' cat', 's'] [16, 22, 12]
['the', ' ', 'b', 'at'] not in vocab: ['b']
11 tokens


## Bytes: the fallback that never fails

The fix is to start from a set of basic symbols that is complete by construction, and computers already have one. Any text in a file or on a network is stored as a sequence of **bytes**, each a number from 0 to 255. The standard rule for turning characters into bytes is called **UTF-8** (Unicode Transformation Format, 8-bit). It gives each plain English letter, digit and punctuation mark one byte, and every other character between two and four.

In [6]:
print(list("café".encode("utf-8")))
print(list("🐈".encode("utf-8")))

[99, 97, 102, 195, 169]
[240, 159, 144, 136]


## A real tokenizer on a real sentence

This series traces one short input, "The cat sat on the", through every stage of a language model, and tokenization is the first stage. Why GPT-2? Its tokenizer is frozen and small enough to inspect. The model you call today has its own tokenizer and will give different counts, so count with that one. The `tiktoken` library ships GPT-2's tokenizer, and the outputs below come from version 0.14.0, the one pinned in the series `requirements.txt` as of October 2026. Its first call downloads two vocabulary files of about 1.5 MB in total, checks each against a hash stored in the library, and caches them.

In [7]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
token_ids = enc.encode("The cat sat on the")
print(token_ids)
print([enc.decode([i]) for i in token_ids])
print(repr(enc.decode(token_ids)))
print(enc.n_vocab, enc.eot_token)

[464, 3797, 3332, 319, 262]
['The', ' cat', ' sat', ' on', ' the']
'The cat sat on the'
50257 50256


The spaces live inside the tokens, as pre-tokenization arranged, so ` cat` with its leading space is token 3797. Change the capital or drop the space and you get a different token.

In [8]:
def pieces(s):
    # one token at a time; a token that is part of a multi-byte character prints as "�"
    return [enc.decode([i]) for i in enc.encode(s)]

for s in [" cat", " Cat", "cat", " CAT"]:
    print(repr(s), enc.encode(s))
for s in ["the bat", "Akwirw ier", "The cat's 2 mats"]:
    print(pieces(s))
print(enc.encode("🐈"))

' cat' [3797]
' Cat' [5181]
'cat' [9246]
' CAT' [38348]
['the', ' bat']
['Ak', 'w', 'ir', 'w', ' ', 'ier']
['The', ' cat', "'s", ' 2', ' mats']
[8582, 238, 230]


## Where tokenization shows through

Most of the time you can ignore the tokenizer. Here are the places an engineer runs into it, all computed with GPT-2's tokenizer.

In [9]:
for s in ["strawberry", " strawberry", " 1234567", " 2026"]:
    print(pieces(s))
for s in ["The cat sat on the mat.", "Katten satt på matten.",
          "Die Katze saß auf der Matte.", "Le chat était assis sur le tapis."]:
    print(len(s.encode("utf-8")), "bytes,", len(enc.encode(s)), "tokens:", s)

['st', 'raw', 'berry']
[' strawberry']
[' 123', '45', '67']
[' 20', '26']
23 bytes, 7 tokens: The cat sat on the mat.
23 bytes, 9 tokens: Katten satt på matten.
29 bytes, 10 tokens: Die Katze saß auf der Matte.
34 bytes, 12 tokens: Le chat était assis sur le tapis.


## From tokens to training examples

Recall from Guide 1 that a training example paired one character with the character that came next. With tokens, the pairing is the same shift by one position.

In [10]:
inputs, targets = token_ids[:-1], token_ids[1:]
print(inputs)
print(targets)

[464, 3797, 3332, 319]
[3797, 3332, 319, 262]


---

← [Guide 1: Neural networks](01-neural-networks.ipynb) · [Guide 3: Embeddings](03-embeddings.ipynb) →

Found a cell that doesn't reproduce the guide? [Open an issue](https://github.com/drvgautam/primer-on-aiengineering/issues/new?template=does-not-reproduce.yml).